# 59 — LSEG/Gemma hybrid intraday versus overnight P&L decomposition

**Question.** Where inside the frozen Notebook 36 open-to-open holding interval does its gross P&L accrue: entry-open to same-session close, or same-session close to the following open?

The design was frozen in `frozen_specs/lseg_gemma_hybrid_session_decomposition_v1.json` before component outcomes were computed. The candidate, Gemma timing/counts, FinBERT ranks, original 33, 25% cap, holding interval, and costs remain unchanged. Start-open-normalized intraday and overnight contributions must add exactly to every Notebook 36 gross return. Costs remain separate because assigning round-trip cost to one component would be arbitrary.

This is a retrospective mechanism and plausibility diagnostic. It does not construct an open-to-close or close-to-open strategy, change the exit, select a component for trading, alter the prospective contract, or promote alpha. No licensed headline text is loaded or emitted.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from statsmodels.stats.multitest import multipletests

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'final_experiments':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from final_experiments.lib.evaluate import annualized_sharpe  # noqa: E402
from final_experiments.lib.plots import CATEGORICAL, INK, apply_house_style  # noqa: E402
from final_experiments.lib.risk_overlay import paired_block_bootstrap_difference  # noqa: E402
from final_experiments.lib.sector_portfolios import SECTOR_MEMBERS  # noqa: E402
from final_experiments.lib.sparse_spread import build_exact_extrema_targets  # noqa: E402

SPEC_PATH = REPO_ROOT / 'final_experiments/frozen_specs/lseg_gemma_hybrid_session_decomposition_v1.json'
AGGREGATE_DIR = REPO_ROOT / 'final_experiments/outputs/13_lseg_44_gemma_robustness'
GEMMA_PATH = AGGREGATE_DIR / 'gemma4_26b_firm_open_aggregators.parquet'
FINBERT_PATH = AGGREGATE_DIR / 'finbert_firm_open_aggregators.parquet'
PRICE_PATH = REPO_ROOT / 'Data/derived/prices/lseg_us_sector_33_8m.csv'
PRICE_MANIFEST_PATH = REPO_ROOT / 'Data/derived/prices/lseg_us_sector_33_8m.manifest.json'
N36_DAILY_PATH = REPO_ROOT / 'final_experiments/outputs/36_lseg_gemma_finbert_hybrid_alpha_audit/capped_hybrid_daily.parquet'
OUTPUT_DIR = REPO_ROOT / 'final_experiments/outputs/59_lseg_gemma_hybrid_session_decomposition'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

spec = json.loads(SPEC_PATH.read_text())
assert spec['status'] == 'frozen_before_component_outcomes'
SEED = int(spec['inference']['seed'])
BLOCK_LENGTH = int(spec['inference']['block_length_sessions'])
REPLICATIONS = int(spec['inference']['replications'])
BH_Q = float(spec['inference']['bh_q'])
SINGLE_NAME_CAP = float(spec['candidate']['single_name_cap'])
IDENTITY_TOLERANCE = float(spec['decomposition']['identity_tolerance'])
apply_house_style()
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')

## Exact target and component reconstruction

For stock $i$ on entry session $t$, the frozen target weight is $w_{i,t}$. The two additive start-open-normalized contributions are

$$r^{ID}_{i,t}=C_{i,t}/O_{i,t}-1,\qquad r^{ON}_{i,t}=(O_{i,t+1}-C_{i,t})/O_{i,t}.$$

Therefore $r^{ID}_{i,t}+r^{ON}_{i,t}=O_{i,t+1}/O_{i,t}-1$ exactly. This is an accounting decomposition of the existing holding interval, not two separately executable returns.

In [ ]:
sector_members = {sector: tuple(members[:3]) for sector, members in SECTOR_MEMBERS.items()}
original_symbols = tuple(symbol for members in sector_members.values() for symbol in members)
symbols = tuple(sorted(original_symbols))
gemma = pd.read_parquet(GEMMA_PATH)
finbert = pd.read_parquet(FINBERT_PATH)
for frame in (gemma, finbert):
    frame['session_date'] = pd.to_datetime(frame['session_date']).dt.normalize()
gemma = gemma.loc[gemma['symbol'].isin(symbols)].copy()
finbert = finbert.loc[finbert['symbol'].isin(symbols)].copy()
if not gemma[['session_date', 'symbol']].equals(finbert[['session_date', 'symbol']]):
    raise ValueError('paired scorer panels are not row-aligned')
if gemma.duplicated(['session_date', 'symbol']).any() or finbert['strongest_event'].isna().any():
    raise ValueError('candidate scorer input is invalid')
sessions = pd.DatetimeIndex(sorted(gemma['session_date'].unique()))
if len(sessions) != 167 or set(gemma['symbol'].unique()) != set(symbols):
    raise ValueError('expected 167 sessions and immutable original 33')

prices = pd.read_csv(PRICE_PATH, parse_dates=['session_date'])
prices['session_date'] = pd.to_datetime(prices['session_date']).dt.normalize()
prices = prices.loc[prices['symbol'].isin(symbols)].copy()
price_manifest = json.loads(PRICE_MANIFEST_PATH.read_text())
assert price_manifest['return_convention'] == 'split-adjusted price returns; dividends not back-adjusted'
assert hashlib.sha256(PRICE_PATH.read_bytes()).hexdigest() == price_manifest['file']['sha256']
open_wide = prices.pivot(index='session_date', columns='symbol', values='open').sort_index().dropna()
close_wide = prices.pivot(index='session_date', columns='symbol', values='close').sort_index().reindex(open_wide.index)
locations = open_wide.index.get_indexer(sessions)
if (locations < 0).any() or not np.array_equal(locations[1:], locations[:-1] + 1):
    raise ValueError('signal sessions are not consecutive complete-price sessions')
if locations[-1] + 1 >= len(open_wide):
    raise ValueError('last signal session has no following open')
current_open = open_wide.loc[sessions, list(symbols)].to_numpy(dtype=float)
current_close = close_wide.loc[sessions, list(symbols)].to_numpy(dtype=float)
next_sessions = open_wide.index[locations + 1]
next_open = open_wide.loc[next_sessions, list(symbols)].to_numpy(dtype=float)
if not np.isfinite(np.stack([current_open, current_close, next_open])).all():
    raise ValueError('component prices must be complete')

def rank_map(frame: pd.DataFrame) -> dict[pd.Timestamp, tuple[str, ...]]:
    result = {
        pd.Timestamp(session): tuple(
            day.sort_values(['strongest_event', 'symbol'], ascending=[False, True], kind='mergesort')['symbol']
        )
        for session, day in frame.groupby('session_date', sort=True)
    }
    if any(set(row) != set(symbols) for row in result.values()):
        raise ValueError('rank map is incomplete')
    return result

_, target_audit = build_exact_extrema_targets(gemma, single_name_cap=SINGLE_NAME_CAP)
if not np.array_equal(pd.to_datetime(target_audit['session_date']).to_numpy(), sessions.to_numpy()):
    raise ValueError('target audit is not aligned to the session spine')
ranks = rank_map(finbert)
symbol_index = {symbol: index for index, symbol in enumerate(symbols)}
weights = np.zeros((len(sessions), len(symbols)), dtype=float)
for row_index, row in enumerate(target_audit.itertuples(index=False)):
    n_long, n_short, eligible = int(row.positive_names), int(row.negative_names), bool(row.eligible)
    if not eligible:
        continue
    ranked = ranks[pd.Timestamp(row.session_date)]
    long_symbols = ranked[:n_long]
    short_symbols = ranked[-n_short:]
    if not n_long or not n_short or set(long_symbols) & set(short_symbols):
        raise RuntimeError('invalid hybrid selection')
    leg = min(0.5, SINGLE_NAME_CAP * n_long, SINGLE_NAME_CAP * n_short)
    for symbol in long_symbols:
        weights[row_index, symbol_index[symbol]] = leg / n_long
    for symbol in short_symbols:
        weights[row_index, symbol_index[symbol]] = -leg / n_short
if np.abs(weights.sum(axis=1)).max() > 1e-12 or np.abs(weights).max() > SINGLE_NAME_CAP + 1e-12:
    raise RuntimeError('target weights violate neutrality or cap')

intraday_stock = current_close / current_open - 1.0
overnight_stock = (next_open - current_close) / current_open
open_to_open_stock = next_open / current_open - 1.0
stock_identity_error = float(np.max(np.abs(intraday_stock + overnight_stock - open_to_open_stock)))
intraday = np.sum(weights * intraday_stock, axis=1)
overnight = np.sum(weights * overnight_stock, axis=1)
gross_recomputed = intraday + overnight
long_mask = weights > 0
short_mask = weights < 0
intraday_long = np.sum(np.where(long_mask, weights * intraday_stock, 0.0), axis=1)
intraday_short = np.sum(np.where(short_mask, weights * intraday_stock, 0.0), axis=1)
overnight_long = np.sum(np.where(long_mask, weights * overnight_stock, 0.0), axis=1)
overnight_short = np.sum(np.where(short_mask, weights * overnight_stock, 0.0), axis=1)

n36 = pd.read_parquet(N36_DAILY_PATH).reset_index(drop=True)
n36['session_date'] = pd.to_datetime(n36['session_date']).dt.normalize()
if not np.array_equal(n36['session_date'].to_numpy(), sessions.to_numpy()):
    raise ValueError('Notebook 36 daily path is not aligned')
portfolio_identity_error = float(np.max(np.abs(gross_recomputed - n36['gross_return'].to_numpy(dtype=float))))
leg_identity_error = float(np.max(np.abs(intraday_long + intraday_short + overnight_long + overnight_short - gross_recomputed)))
if max(stock_identity_error, portfolio_identity_error, leg_identity_error) > IDENTITY_TOLERANCE:
    raise RuntimeError('session decomposition identity failed')

daily = pd.DataFrame({
    'session_date': sessions, 'return_end_date': next_sessions,
    'intraday_gross_contribution': intraday, 'overnight_gross_contribution': overnight,
    'gross_return_recomputed': gross_recomputed,
    'intraday_long': intraday_long, 'intraday_short': intraday_short,
    'overnight_long': overnight_long, 'overnight_short': overnight_short,
    'cost': n36['cost'].to_numpy(dtype=float), 'net_return': n36['net_return'].to_numpy(dtype=float),
    'gross_exposure': n36['gross_exposure'].to_numpy(dtype=float),
})
daily['active'] = daily['gross_exposure'].gt(0)
identity = pd.DataFrame([{
    'stock_component_max_error': stock_identity_error,
    'portfolio_vs_notebook36_max_error': portfolio_identity_error,
    'leg_component_max_error': leg_identity_error,
    'sessions': len(daily), 'active_sessions': int(daily['active'].sum()),
    'companies': len(symbols), 'licensed_headline_text_loaded': False,
}])
display(identity.T)

## Frozen two-component inference family

Both component means are tested against zero over all 167 complete sessions using the same five-session circular-block procedure as Notebook 36. Benjamini–Hochberg controls the two declared component tests. Conditional-active means, halves, and long/short pieces are descriptive diagnostics rather than additional tests.

In [ ]:
component_columns = {
    'intraday': 'intraday_gross_contribution',
    'overnight': 'overnight_gross_contribution',
}
cash = daily[['session_date']].copy()
cash['value'] = 0.0
inference_rows = []
split = (len(daily) + 1) // 2
total_gross_sum = float(daily['gross_return_recomputed'].sum())
for component, column in component_columns.items():
    challenger = daily[['session_date', column]].rename(columns={column: 'value'})
    inference = paired_block_bootstrap_difference(
        challenger, cash, value_col='value', block_length=BLOCK_LENGTH,
        replications=REPLICATIONS, seed=SEED,
    )
    values = daily[column].to_numpy(dtype=float)
    inference_rows.append({
        'component': component,
        'mean_bps_session': float(values.mean() * 10_000),
        'ci_low_bps_session': float(inference['ci_low'] * 10_000),
        'ci_high_bps_session': float(inference['ci_high'] * 10_000),
        'p_two_sided': float(inference['p_two_sided']),
        'sharpe_all_sessions': float(annualized_sharpe(pd.Series(values))),
        'mean_bps_active_session': float(daily.loc[daily['active'], column].mean() * 10_000),
        'first_half_sharpe': float(annualized_sharpe(pd.Series(values[:split]))),
        'second_half_sharpe': float(annualized_sharpe(pd.Series(values[split:]))),
        'arithmetic_total_contribution': float(values.sum()),
        'share_of_total_gross_sum': float(values.sum() / total_gross_sum),
    })
inference_table = pd.DataFrame(inference_rows)
reject, q_values, _, _ = multipletests(inference_table['p_two_sided'], alpha=BH_Q, method='fdr_bh')
inference_table['bh_q_value'] = q_values
inference_table['bh_reject'] = reject
indexed = inference_table.set_index('component')
intraday_pass = bool(indexed.loc['intraday', 'mean_bps_session'] > 0 and indexed.loc['intraday', 'bh_reject'])
overnight_pass = bool(indexed.loc['overnight', 'mean_bps_session'] > 0 and indexed.loc['overnight', 'bh_reject'])
mechanism_class = (
    'both' if intraday_pass and overnight_pass else
    'intraday_only' if intraday_pass else
    'overnight_only' if overnight_pass else
    'neither'
)
leg_totals = pd.DataFrame([
    {'segment': 'intraday', 'leg': 'long', 'arithmetic_total_contribution': float(daily['intraday_long'].sum())},
    {'segment': 'intraday', 'leg': 'short', 'arithmetic_total_contribution': float(daily['intraday_short'].sum())},
    {'segment': 'overnight', 'leg': 'long', 'arithmetic_total_contribution': float(daily['overnight_long'].sum())},
    {'segment': 'overnight', 'leg': 'short', 'arithmetic_total_contribution': float(daily['overnight_short'].sum())},
])
economics = pd.DataFrame([{
    'gross_mean_bps_session': float(daily['gross_return_recomputed'].mean() * 10_000),
    'cost_mean_bps_session': float(daily['cost'].mean() * 10_000),
    'net_mean_bps_session': float(daily['net_return'].mean() * 10_000),
    'gross_sharpe': float(annualized_sharpe(daily['gross_return_recomputed'])),
    'net_sharpe': float(annualized_sharpe(daily['net_return'])),
    'mechanism_class': mechanism_class,
}])
display(inference_table)
display(leg_totals)
display(economics.T)

## Results, plots, and persisted evidence

Cumulative component lines use arithmetic gross contributions because the two pieces are additive by construction; presenting separately compounded wealth paths would break the identity.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.0))
axes[0, 0].plot(daily['session_date'], 100 * daily['gross_return_recomputed'].cumsum(), color=INK['reference'], lw=2.2, label='Open-to-open gross')
axes[0, 0].plot(daily['session_date'], 100 * daily['intraday_gross_contribution'].cumsum(), color=CATEGORICAL[0], lw=1.8, label='Intraday component')
axes[0, 0].plot(daily['session_date'], 100 * daily['overnight_gross_contribution'].cumsum(), color=CATEGORICAL[1], lw=1.8, label='Overnight component')
axes[0, 0].axhline(0, color=INK['reference'], lw=0.8)
axes[0, 0].set_ylabel('Cumulative arithmetic contribution (%)')
axes[0, 0].set_title('Exact additive path decomposition')
axes[0, 0].legend(frameon=False, fontsize=8)
x = np.arange(len(inference_table))
means = inference_table['mean_bps_session'].to_numpy(dtype=float)
lower = means - inference_table['ci_low_bps_session'].to_numpy(dtype=float)
upper = inference_table['ci_high_bps_session'].to_numpy(dtype=float) - means
axes[0, 1].bar(x, means, color=CATEGORICAL[:2], alpha=0.85)
axes[0, 1].errorbar(x, means, yerr=np.vstack([lower, upper]), fmt='none', color=INK['reference'], capsize=5)
axes[0, 1].axhline(0, color=INK['reference'], lw=1)
axes[0, 1].set_xticks(x, inference_table['component'].str.title())
axes[0, 1].set_ylabel('Mean gross contribution (bps/session)')
axes[0, 1].set_title('Five-session block-bootstrap intervals')
leg_pivot = leg_totals.pivot(index='segment', columns='leg', values='arithmetic_total_contribution').loc[['intraday', 'overnight']]
leg_pivot.plot(kind='bar', ax=axes[1, 0], color=[CATEGORICAL[2], CATEGORICAL[3]], rot=0)
axes[1, 0].axhline(0, color=INK['reference'], lw=1)
axes[1, 0].set_ylabel('Arithmetic total gross contribution')
axes[1, 0].set_xlabel('')
axes[1, 0].set_title('Long and short leg attribution')
axes[1, 0].legend(title='Leg', frameon=False)
width = 0.36
axes[1, 1].bar(x - width / 2, inference_table['first_half_sharpe'], width, color=CATEGORICAL[4], label='First half')
axes[1, 1].bar(x + width / 2, inference_table['second_half_sharpe'], width, color=CATEGORICAL[5], label='Second half')
axes[1, 1].axhline(0, color=INK['reference'], lw=1)
axes[1, 1].set_xticks(x, inference_table['component'].str.title())
axes[1, 1].set_ylabel('Gross component Sharpe')
axes[1, 1].set_title('Temporal stability is descriptive')
axes[1, 1].legend(frameon=False)
fig.suptitle('Frozen LSEG/Gemma-FinBERT hybrid: where open-to-open gross P&L accrues')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'session_component_decomposition.png', dpi=180, bbox_inches='tight')
plt.show()

identity.to_csv(OUTPUT_DIR / 'identity.csv', index=False)
inference_table.to_csv(OUTPUT_DIR / 'component_inference.csv', index=False)
leg_totals.to_csv(OUTPUT_DIR / 'leg_component_totals.csv', index=False)
economics.to_csv(OUTPUT_DIR / 'economics.csv', index=False)
daily.to_parquet(OUTPUT_DIR / 'daily_component_audit.parquet', index=False)
decision = {
    'mechanism_class': mechanism_class,
    'intraday_bh_significant_positive': intraday_pass,
    'overnight_bh_significant_positive': overnight_pass,
    'intraday_share_of_total_gross_sum': float(indexed.loc['intraday', 'share_of_total_gross_sum']),
    'overnight_share_of_total_gross_sum': float(indexed.loc['overnight', 'share_of_total_gross_sum']),
    'alternative_exit_return_constructed': False,
    'strategy_changed': False,
    'prospective_contract_changed': False,
    'validated_alpha': False,
    'deployment_qualified': False,
}
try:
    git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = None
manifest = {
    'status': 'RETROSPECTIVE_OPENED_WINDOW_SESSION_COMPONENT_DECOMPOSITION',
    'notebook': '59_lseg_gemma_hybrid_session_decomposition.ipynb',
    'git_commit_at_execution': git_commit,
    'frozen_spec_sha256': hashlib.sha256(SPEC_PATH.read_bytes()).hexdigest(),
    'specification': spec,
    'input_boundary': {
        'gemma_aggregate_panel': str(GEMMA_PATH.relative_to(REPO_ROOT)),
        'finbert_aggregate_panel': str(FINBERT_PATH.relative_to(REPO_ROOT)),
        'price_path': str(PRICE_PATH.relative_to(REPO_ROOT)),
        'notebook36_daily': str(N36_DAILY_PATH.relative_to(REPO_ROOT)),
        'licensed_headline_text_loaded': False, 'licensed_text_emitted': False,
        'source_regimes_pooled': False, 'row_level_component_audit_persisted_local_only': True,
    },
    'identity': json.loads(identity.to_json(orient='records'))[0],
    'component_inference': json.loads(inference_table.to_json(orient='records')),
    'leg_component_totals': json.loads(leg_totals.to_json(orient='records')),
    'economics': json.loads(economics.to_json(orient='records'))[0],
    'decision': decision,
    'claim_boundary': {
        'new_signal_constructed': False, 'alternative_exit_return_constructed': False,
        'strategy_reselected': False, 'historical_retuning_performed': False,
        'prospective_contract_changed': False, 'alpha_promotion_permitted': False,
    },
    'limitations': [
        'The candidate and return window were selected before this decomposition.',
        'Intraday and overnight pieces are start-open-normalized accounting contributions, not separately executable strategy returns.',
        'Round-trip costs cannot be uniquely assigned to one gross component and remain separate.',
        'Daily bars do not identify the intraday path, auction slippage, spreads, borrow, financing, or order-book depth.',
        'Split-adjusted price returns exclude dividends.',
        'Human validation of full-corpus Gemma sentiment and genuinely new-date replay remain required.',
    ],
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')

intraday_row = indexed.loc['intraday']
overnight_row = indexed.loc['overnight']
display(Markdown(
    '### Decision\n\n'
    f"- The exact gross-return identity passes with maximum error **{portfolio_identity_error:.2e}**.\n"
    f"- Intraday contributes **{intraday_row['mean_bps_session']:+.2f} bps/session** "
    f"[{intraday_row['ci_low_bps_session']:+.2f}, {intraday_row['ci_high_bps_session']:+.2f}], "
    f"BH q=**{intraday_row['bh_q_value']:.4f}**.\n"
    f"- Overnight contributes **{overnight_row['mean_bps_session']:+.2f} bps/session** "
    f"[{overnight_row['ci_low_bps_session']:+.2f}, {overnight_row['ci_high_bps_session']:+.2f}], "
    f"BH q=**{overnight_row['bh_q_value']:.4f}**.\n"
    f"- Frozen mechanism classification: **{mechanism_class}**. This does not change the open-to-open strategy or validate alpha."
))
print(json.dumps(decision, indent=2))

## Next steps

1. Preserve the decomposition even if neither component survives the two-test family.
2. Do not turn the stronger component into a new exit rule on this opened window.
3. Keep the exact Notebook 36 open-to-open candidate for the one frozen prospective replay.
4. Revisit session attribution only with genuinely new dates or independently sourced intraday execution data.